# **Building an End-to-End RAG Chain**

**Step 1: Initialize the Chroma DB Connection**  
**Step 2: Create a Retriever Object**   
**Step 3: Initialize a Chat Prompt Template**  
**Step 4: Initialize a Generator (i.e. Chat Model)**  
**Step 5: Initialize a Output Parser**   
**Step 6: Define a RAG Chain**  
**Step 7: Invoke the Chain**

## **Step 1: Initialize the Chroma DB Connection**

In [1]:
# Initialize a ChromaDB Connection

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-small-en-v1.5"

hf_bge_embd_model = HuggingFaceEmbeddings(model_name=model_name)

# Initialize the database connection
# If database exist, it will connect with the collection_name and persist_directory
# Otherwise a new collection will be created
vectordb = Chroma(
    persist_directory="./chroma_db_hf_bge",
    collection_name="vector_database", 
    embedding_function=hf_bge_embd_model, 
)

# We can check the already existing values
print(f"Number of Embeddings inside the Vector DB: {len(vectordb.get()["ids"])}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Number of Embeddings inside the Vector DB: 295


## **Step 2: Create a Retriever Object**

In [2]:
retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [3]:
chunks = retriever.invoke("who is rachem?")

print(len(chunks))

print(chunks)

5
[Document(id='a501a474-0d3f-4635-ad0e-e5899dc027ec', metadata={'source': 'data\\subtitles\\Friends_2x09.srt'}, page_content="109\n00:06:58,220 --> 00:07:01,712\nAll right. So he's not\na famous tree surgeon?\n\n110\n00:07:06,128 --> 00:07:10,462\nAnd I guess he doesn't live in a hut\nin Burma where there's no phones?\n\n111\n00:07:11,266 --> 00:07:14,360\nLast I heard, he was\na pharmacist somewhere upstate.\n\n112\n00:07:14,570 --> 00:07:16,197\nThat makes no sense.\n\n113\n00:07:16,405 --> 00:07:19,465\nWhy would the villagers\nworship a pharmacist?\n\n114\n00:07:21,476 --> 00:07:22,738\nHoney.\n\n115\n00:07:26,148 --> 00:07:29,140\nAnyway, that's all I know.\n\n116\n00:07:31,720 --> 00:07:32,812\nThat...\n\n117\n00:07:34,990 --> 00:07:36,457\n...and this.\n\n118\n00:07:39,795 --> 00:07:40,887\nThis...\n\n119\n00:07:41,830 --> 00:07:43,730\n...is the real him.\n\n120\n00:07:48,770 --> 00:07:51,898\nI remember my father\nall dressed up in the red suit...\n\n121\n00:07:52,140 --> 00:

## **Step 3: Initialize a Chat Prompt Template**

In [4]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate(
    messages=[
        ("system", """
        You are a knowledgeable and reliable AI assistant.

        Your task is to answer the user's question ONLY using the provided context.
        
        Instructions:
        - Carefully read all of the retrieved context before answering.
        - Base your answer entirely on the provided context.
        - Do not invent, assume, or infer facts that are not supported by the context.
        - If the context does not contain enough information to answer the question, clearly state:
          "I don't have enough information in the provided context to answer that."
        - If multiple pieces of context are relevant, combine them into one coherent answer.
        - Ignore duplicated information.
        - If the context contains conflicting information, mention the conflict instead of choosing one.
        - Keep the answer concise while including all important details.
        - When appropriate, answer using bullet points or numbered lists.
        - Never mention internal implementation details such as embeddings, vector databases, retrieval, chunks, or similarity scores.
        
        Context:
        {context}
        """), 
        ("human", """Question:
        {question}
        """),
    ]
)

## **Step 4: Initialize a Generator (i.e. Chat Model)**

In [5]:
from langchain_groq import ChatGroq

with open("keys/.groq_api_key.txt") as f:
    GROQ_API_KEY = f.read()

groq_chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="openai/gpt-oss-120b", 
    temperature=1
)

groq_chat_model.invoke("hi").content

'Hello! How can I help you today?'

## **Step 5: Initialize a Output Parser**

In [6]:
# TODO
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

## **Step 6: Define RAG Chain**

1. Init a base chain
2. Define chunk merging logic
3. Define a RAG Chain

In [7]:
# Init a base chain

base_chain = template | groq_chat_model | parser

In [8]:
lst_of_chunks = ["hi","How are you","what's up?"]

In [9]:
def func(lst_of_chunks):
    return "\n\n".join(chunk for chunk in lst_of_chunks)

(func(lst_of_chunks))

"hi\n\nHow are you\n\nwhat's up?"

In [10]:
print(func(lst_of_chunks))

hi

How are you

what's up?


In [11]:
def func_1(x):
    return x+x

def func_2(y):
    return y+y

In [12]:
# chain = func_1 | func_2

# ---------------------------------------------------------------------------
# TypeError                                 Traceback (most recent call last)
# Cell In[17], line 1
# ----> 1 chain = func_1 | func_2

# TypeError: unsupported operand type(s) for |: 'function' and 'function'

#you can not use function inside the chain to run it requires the runnablelambda

In [13]:
from langchain_core.runnables import RunnableLambda

run_func_1 = RunnableLambda(lambda x : func_1(x))

run_func_2 = RunnableLambda(lambda y : func_2(y))

chain = run_func_1 | run_func_2

chain.invoke(2)

8

In [14]:
# Define chunk merging logic

from langchain_core.runnables import RunnableLambda

def format_chunks(chunks):
    return "\n\n".join(chunk.page_content for chunk in chunks)

runnable_format_chunks = RunnableLambda(lambda x : format_chunks(x))

#### **Important:**
- base_chain takes two inputs
- context -> retriever + runnable_format_docs
- question -> user provides this

In [15]:
# Define a RAG Chain
from langchain_core.runnables import RunnablePassthrough

rag_chain = {
    "context": retriever | runnable_format_chunks, 
    "question": RunnablePassthrough()
} | base_chain

In [16]:
# rag_chain = RunnableParallel(
#     context=retriever | runnable_format_docs,
#     question=RunnablePassthrough
# ) | base_chain

## **Step 7: Invoke the Chain**

In [17]:
# Invoke the Chain

query = 'Who is Rachem?'

response = rag_chain.invoke(query)

print(response)

I don't have enough information in the provided context to answer that.


In [18]:
# Invoke the Chain

query = 'What is there on the List comparing Rachel and Julie?'

response = rag_chain.invoke(query)

print(response)

**List comparing Rachel and Julie (pros / cons as shown in the dialogue)**  

- **Rachel (cons)**  
  - “a little spoiled sometimes”  
  - “a little ditzy”  
  - “a little too into her looks”  
  - “her ankles are a little chubby”  

- **Julie (pros/notes)**  
  - “we’re both paleontologists” – Julie (and the speaker) share the same profession  
  - “She’s not Rachel” – implying Julie is different from Rachel (i.e., not a waitress)  

The list is presented as a “pros and cons” comparison, but the dialogue focuses mainly on the cons for Rachel and a few positive points for Julie.
